In [1]:
import os
import sys
sys.path.append('/Users/antonio/Desktop/DataScience/MyCode/langchain-pdf')

In [2]:
from dotenv import find_dotenv
from dotenv import load_dotenv


load_dotenv(find_dotenv(), override=True)

True

In [3]:
def _process_kwargs(kwargs: dict) -> dict:
    if "functor" in kwargs and isinstance(kwargs["functor"], list):
        functor_list = []
        for functor in kwargs["functor"]:
            if isinstance(functor, list):
                functor_list.append((functor[0], functor[1]))
            else:
                functor_list.append(functor)
        kwargs["functor"] = functor_list
    return kwargs

In [4]:
import os
from functools import cached_property

import yaml  # type: ignore



class ChatConfig:
    def __init__(self, config_file: str):
        with open(config_file) as f:
            self._yaml_data = yaml.safe_load(f)

    @cached_property
    def splitter_map(self):
        return self._build_map("text_splitter")

    @cached_property
    def llm_map(self):
        return self._build_map("llm")

    @cached_property
    def embedding_map(self):
        return self._build_embeddings()

    @cached_property
    def vector_store_map(self):
        return self._build_vector_store_map()

    @cached_property
    def vector_stores(self):
        return self._build_vector_store_list()

    @cached_property
    def retriever_map(self):
        return self._build_map("retriever")

    @cached_property
    def memory_map(self):
        return self._build_map("memory")

    @cached_property
    def condense_question_llm_kwargs(self):
        chain_config = self._yaml_data.get("chain", {})
        return chain_config.get("condense_question_llm", {})

    def _init_component(self, component: dict):
        env_variables = component.get("env", {})
        for key, value in env_variables.items():
            print(f"Setting environment variable: {key} = {value}")
        shell_commands = component.get("shell", [])
        for command in shell_commands:
            print(f"Executing shell command: {command}")

    def _build_map(self, component_type: str) -> dict:
        print("-" * 50)
        print(f"Building map for: {component_type}")
        component_map = {}
        for component in self._yaml_data[component_type]:
            print(">" * 50)
            component_name = component["name"]
            component_module = component["module"]
            component_builder = component["builder"]
            component_kwargs = component.get("params", {})
            component_map[component_name] = (component_module, component_builder, component_kwargs)
            print(f"Name: {component_name}")
            print(f"Module: {component_module}")
            print(f"Builder: {component_builder}")
            print(f"Kwargs: {component_kwargs}")
            self._init_component(component)
            print("<" * 50)
        print("-" * 50)
        return component_map

    def _build_embeddings(self) -> dict:
        print("-" * 50)
        print(f"Building Embedding map")
        embedding_map = {}
        for embedding in self._yaml_data["embedding"]:
            print(">" * 50)
            self._init_component(embedding)
            embedding_name = embedding["name"]
            embedding_model = embedding["model"]
            embedding_kwargs = embedding.get("params", {})
            print(f"Name: {embedding_name}")
            print(f"Model: {embedding_model}")
            print(f"Kwargs: {embedding_kwargs}")
            embedding_map[embedding_name] = (embedding_model, embedding_kwargs)
            self._init_component(embedding)
            print("<" * 50)
        print("-" * 50)
        return embedding_map

    def _build_vector_store_map(self) -> dict:
        print("-" * 50)
        print(f"Building Vectorstore map")
        vector_store_map = {}
        for splitter_name in self.splitter_map.keys():
            store_map_level2 = {}
            for vector_store in self._yaml_data["vector_store"]:
                self._init_component(vector_store)
                vector_store_name = vector_store["name"]
                vector_store_module = vector_store["module"]
                vector_store_builder = vector_store["builder"]
                vector_store_kwargs = _process_kwargs(vector_store.get("params", {}))
                store_map_level3 = {}
                for embedding_name, embedding in self.embedding_map.items():
                    print(">" * 50)
                    print(f"Name: {vector_store_name}")
                    print(f"Module: {vector_store_module}")
                    print(f"Builder: {vector_store_builder}")
                    print(f"Kwargs: {vector_store_kwargs}")
                    print(f"Spliter: {splitter_name}")
                    print(f"Embedding: {embedding_name}")
                    store = (vector_store_module, vector_store_builder, splitter_name, embedding_name, vector_store_kwargs)
                    store_map_level3[embedding_name] = store
                    print("<" * 50)
                store_map_level2[vector_store_name] = store_map_level3
            vector_store_map[splitter_name] = store_map_level2
        return vector_store_map

    def _build_vector_store_list(self) -> dict:
        used_vector_stores = {}
        for retriever in self._yaml_data["retriever"]:
            vector_store_name = retriever["module"].split(".")[-1]
            retriever_params = retriever.get("params", {})
            embedding_name = retriever_params["embedding_name"]
            splitter_name = retriever_params["splitter_name"]
            if splitter_name not in used_vector_stores:
                used_vector_stores[splitter_name] = []
            used_vector_stores[splitter_name].append(
                self.vector_store_map[splitter_name][vector_store_name][embedding_name]
            )
        return used_vector_stores


chat_config = ChatConfig("./app/chat/config.yaml")


In [5]:
chat_config.vector_store_map

--------------------------------------------------
Building Vectorstore map
--------------------------------------------------
Building map for: text_splitter
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Name: recursive_character
Module: app.chat.text_splitters.recursive
Builder: recursive_character_text_splitter_builder
Kwargs: {'chunk_size': 4000, 'chunk_overlap': 200}
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
--------------------------------------------------
--------------------------------------------------
Building Embedding map
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Name: openai_small
Model: OpenAI
Kwargs: {'model_name': 'text-embedding-3-small'}
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Name: openai_large
Model: OpenAI
Kwargs: {'model_name': 'text-embedding-3-large'}
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
--------------------------------------------------
>>>>>>>>>>>>>>>>>>>>>>>>>

{'recursive_character': {'chroma': {'openai_small': ('app.chat.vector_stores.chroma',
    'chroma_vector_store_builder',
    'recursive_character',
    'openai_small',
    {}),
   'openai_large': ('app.chat.vector_stores.chroma',
    'chroma_vector_store_builder',
    'recursive_character',
    'openai_large',
    {})},
  'multi_vector': {'openai_small': ('app.chat.vector_stores.multi_vector',
    'multi_vector_store_builder',
    'recursive_character',
    'openai_small',
    {'functor': ['summary', ('question', {'q': 2})],
     'llm': {'model_name': 'gpt-4o-mini'},
     'add_originals': True}),
   'openai_large': ('app.chat.vector_stores.multi_vector',
    'multi_vector_store_builder',
    'recursive_character',
    'openai_large',
    {'functor': ['summary', ('question', {'q': 2})],
     'llm': {'model_name': 'gpt-4o-mini'},
     'add_originals': True})}}}